# LangChain Memory 시리즈 4/7: ConversationEntityMemory

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요.

- 엔티티(Entity) 기반 메모리가 왜 필요한지 설명할 수 있어요
- LLM이 대화에서 인물·장소·조직 등 엔티티를 자동 추출하는 방식을 이해할 수 있어요
- `entity_store.store`에서 추출된 엔티티 정보를 조회할 수 있어요
- 단순 버퍼 메모리 대비 엔티티 메모리의 장점을 설명할 수 있어요

## 사전 지식

| 개념 | 설명 |
|------|------|
| 엔티티 (Entity) | 이름이 있는 실세계의 객체 (인물, 장소, 조직 등) |
| NER (Named Entity Recognition, 개체명 인식) | 텍스트에서 엔티티를 자동으로 찾아내는 기법 |
| ConversationBufferMemory | 전체 대화를 순서대로 저장하는 기본 메모리 (1번 노트북) |

## 왜 엔티티 메모리가 필요할까요?

버퍼 메모리는 대화 전체를 저장해요. 대화가 길어지면 토큰을 많이 사용하고, 특정 인물에 대한 정보를 빠르게 찾기도 어려워요.

`ConversationEntityMemory`는 LLM을 활용해 대화에서 **엔티티를 추출**하고, 엔티티별로 정보를 구조화해서 저장해요. "민수"에 대한 정보가 궁금하면 전체 대화를 다시 읽을 필요 없이 `entity_store["민수"]`만 조회하면 돼요.

```mermaid
flowchart LR
    CONV["대화: 민수는 백엔드 개발자이고<br/>파이썬을 전문으로 해요"]
    CONV --> LLM_EXT[LLM 엔티티 추출]
    LLM_EXT --> STORE["entity_store<br/>{'민수': '백엔드 개발자, 파이썬 전문'}"]

    CONV2["대화: 민수는 테크스타트에서 일해요"]
    CONV2 --> LLM_EXT2[LLM 엔티티 추출]
    LLM_EXT2 --> STORE2["entity_store 업데이트<br/>{'민수': '백엔드 개발자, 파이썬 전문,<br/>테크스타트 소속'}"]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    class CONV,CONV2 input
    class LLM_EXT,LLM_EXT2 process
    class STORE,STORE2 storage
```

> **참고**: 이 노트북은 레거시 메모리 클래스를 사용해요. LangChain 1.0.x에서는 엔티티 추출 도구 + `Store` 패턴이 권장돼요. 최신 방식은 14번 노트북을 참고하세요.

In [ ]:
# ---------------------------------------------------
# 환경 변수 로드 및 엔티티 메모리 모듈 임포트
# ---------------------------------------------------
from dotenv import load_dotenv
# ⚠️ LangChain 1.0.x에서는 더 이상 `langchain.memory` 패키지를 사용하지 않습니다.
#    구식 메모리 클래스를 시연하기 위해 `langchain.memory`에서 불러옵니다.
from langchain.memory import ConversationEntityMemory
# ChatOpenAI: 엔티티 추출에 사용되는 LLM 인스턴스
from langchain_openai import ChatOpenAI

load_dotenv()

## 1. 기본 사용법 - 엔티티 정보 추출 및 저장

`ConversationEntityMemory`는 LLM을 사용하여 대화에서 엔티티 정보를 추출합니다.


### 💡 엔티티 메모리 초기화

```mermaid
graph TD
    LOAD_ENV[load_dotenv()] --> LLM_INIT[ChatOpenAI(model="gpt-4o-mini", T=0)]
    LLM_INIT --> MEM_CREATE[ConversationEntityMemory(llm=llm)]
    MEM_CREATE --> READY[엔티티 추출 준비 완료]
```



In [ ]:
# ============================================================
# TODO: ConversationEntityMemory를 생성하세요
# 힌트: ChatOpenAI(model="gpt-4o-mini", temperature=0) → ConversationEntityMemory(llm=llm)
# 예상 결과: save_context() 호출 시 LLM이 자동으로 엔티티를 추출해 entity_store에 저장
# ============================================================

# 1단계: LLM 생성 (엔티티 추출에 사용)
# - temperature=0: 엔티티 추출 결과의 일관성 확보
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2단계: ConversationEntityMemory 생성
# - llm 파라미터: 대화에서 엔티티와 속성을 인식하는 LLM (필수)


### 💡 대화 저장 → 엔티티 추출 흐름

```mermaid
flowchart TD
    H1[사용자 발화] --> SAVE1[save_context()]
    SAVE1 --> HIST1[history 저장]
    HIST1 --> CALL1[LLM 호출로 엔티티 추출]
    CALL1 --> STORE1[entity_store.store 업데이트]
    STORE1 --> QUERY1[엔티티별 정보 누적]
```



### 대화 저장 및 엔티티 추출

대화를 저장하면 엔티티 정보가 자동으로 추출되어 저장됩니다.


In [ ]:
# ============================================================
# TODO: 엔티티 정보가 포함된 대화를 save_context()로 저장하세요
# 힌트: save_context(inputs={"human": "..."}, outputs={"ai": "..."})
# 예상 결과: memory.entity_store.store에 민수, 지영 등의 엔티티가 딕셔너리로 저장됨
# ============================================================

# 엔티티 정보(인물, 관계, 기술 스택)가 포함된 대화 저장
# - save_context() 호출 시 LLM이 자동으로 엔티티를 추출하여 entity_store에 기록


### 저장된 엔티티 확인

`entity_store.store`에서 추출된 엔티티 정보를 확인할 수 있습니다.


### 💡 엔티티 정보 누적 구조

```mermaid
flowchart LR
    subgraph Updates[추가 대화 입력]
        H2[새 메시지] --> SAVE2[save_context()]
        SAVE2 --> HIST2[history 누적]
        HIST2 --> LLM2[LLM 엔티티 추출]
    end

    LLM2 --> STORE2[entity_store.store]
    STORE2 --> VIEW2[엔티티별 요약 출력]
    STORE2 --> MERGE2[기존 엔티티 정보와 병합]
```



In [ ]:
# ---------------------------------------------------
# entity_store.store에서 추출된 엔티티 조회
# ---------------------------------------------------
# entity_store.store: 엔티티 이름을 키, 요약 정보를 값으로 갖는 딕셔너리


### 💡 메모리 변수 로드 구조

```mermaid
flowchart TD
    HIST3[history]
    STORE3[entity_store.store]

    QUERY3["load_memory_variables({})"]
    HIST3 -.-> QUERY3
    STORE3 -.-> QUERY3

    QUERY3 --> OUTPUT_ENT["'entities' 키"]
    QUERY3 --> OUTPUT_HIST["'history' 키"]
    OUTPUT_ENT --> PRINT_ENT[엔티티 요약 출력]
    OUTPUT_HIST --> PRINT_HIST[대화 기록 출력]
```



### 추가 대화로 엔티티 정보 축적

더 많은 대화를 저장하면 엔티티 정보가 점진적으로 축적됩니다.


### 💡 프로젝트 관리 시나리오 흐름

```mermaid
flowchart TD
    subgraph ProjectChat
        STEP1[팀 구성 정보 입력] --> SAVE_P1[save_context]
        STEP2[프로젝트 정보 입력] --> SAVE_P2[save_context]
        STEP3[역할 분담 입력] --> SAVE_P3[save_context]
        STEP4[팀장 역할 입력] --> SAVE_P4[save_context]
    end

    SAVE_P1 & SAVE_P2 & SAVE_P3 & SAVE_P4 --> LLM_P[엔티티 추출 LLM]
    LLM_P --> STORE_P[project_memory.entity_store]
    STORE_P --> VIEW_P[엔티티별 정보 확인]
```



In [ ]:
# ============================================================
# TODO: 추가 대화 2개를 저장하여 엔티티 정보를 업데이트하세요
# 힌트: save_context()를 2회 호출 → entity_store.store 재조회
# 예상 결과: 민수/지영의 기술 스택과 "테크스타트" 엔티티가 추가/업데이트됨
# ============================================================

# 1단계: 기술 스택 정보 추가 (기존 엔티티에 새 속성이 병합됨)


# 2단계: 스타트업 이름 정보 추가


# 업데이트된 엔티티 정보 확인


## 2. 메모리 변수 로드

`load_memory_variables()`를 사용하여 저장된 엔티티 정보와 대화 기록을 확인할 수 있습니다.


### 💡 엔티티 정보를 활용한 응답 구성

```mermaid
flowchart LR
    STORE_P2[project_memory.entity_store] --> LOAD_P["load_memory_variables({})"]
    HIST_P[project_memory.history] --> LOAD_P
    LOAD_P --> ENT_OUT[entities 문자열]
    LOAD_P --> HIST_OUT[history 문자열]
    ENT_OUT --> PROMPT_P[프롬프트에 엔티티 정보 삽입]
    HIST_OUT --> PROMPT_P
    PROMPT_P --> ANSWER_P[맥락 유지 응답 생성]
```



In [ ]:
# ---------------------------------------------------
# load_memory_variables()로 entities와 history 동시 조회
# ---------------------------------------------------
# ⚠️ ConversationEntityMemory의 레거시 버그: 빈 딕셔너리({}) 전달 시 KeyError 발생
#    반드시 input_key를 포함한 딕셔너리 형태로 호출해야 함
# memory_variables = memory.load_memory_variables({})  # KeyError 발생!

# input_key를 포함하여 호출 (더미 값 사용)


## 3. 실전 예제: 프로젝트 관리 챗봇

프로젝트 관리 시나리오에서 팀원, 프로젝트, 작업 등의 엔티티 정보를 추적하는 예제입니다.


## 4. 엔티티 정보 활용

저장된 엔티티 정보를 활용하여 대화 맥락을 유지할 수 있습니다.


In [ ]:
# ---------------------------------------------------
# 엔티티 정보를 포함한 메모리 변수 로드 및 활용
# ---------------------------------------------------
# ⚠️ 레거시 버그 우회: input_key 포함 필수
# memory_vars = project_memory.load_memory_variables({})  # KeyError 발생!


## 핵심 정리

```python
# ConversationEntityMemory 기본 사용법
from langchain.memory import ConversationEntityMemory
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
memory = ConversationEntityMemory(llm=llm)

# 대화 저장 (엔티티 자동 추출)
memory.save_context(
    inputs={"human": "사용자 메시지"},
    outputs={"ai": "AI 응답"}
)

# 엔티티 정보 확인
entities = memory.entity_store.store

# 메모리 변수 로드
memory_vars = memory.load_memory_variables({})
```

### 주요 특징
- ✅ **엔티티 추출**: LLM을 사용하여 대화에서 엔티티 정보 자동 추출
- ✅ **지식 축적**: 시간이 지나면서 엔티티 정보를 점진적으로 축적
- ✅ **구조화된 저장**: 엔티티별로 정보를 구조화하여 저장
- ✅ **효율적 메모리**: 엔티티 정보를 별도로 관리하여 효율적인 메모리 사용

### 추출 가능한 엔티티 유형
- 👤 **인물**: 이름, 역할, 관계 등
- 🏢 **조직**: 회사명, 부서, 프로젝트 등
- 📍 **장소**: 위치, 주소 등
- 📅 **날짜/시간**: 일정, 기간 등
- 🎯 **기타**: 제품, 서비스, 개념 등

### 주의사항
- ⚠️ **LLM 필요**: 엔티티 추출을 위해 LLM 인스턴스가 필요함
- ⚠️ **추출 정확도**: LLM의 성능에 따라 엔티티 추출 정확도가 달라짐
- ⚠️ **비용**: 엔티티 추출에 LLM 호출이 필요하므로 비용 발생
- ⚠️ **대화형 사용**: ConversationChain과 함께 사용하는 것이 일반적 (deprecated 경고 있음)